# Final quality-equivalent serving audit

This is the strict current-checkpoint comparison. Both models use **BF16 inference**, **batch=1**, **the exact canonical last-200 semantics**, `torch.compile` where supported, and CUDA Graph replay on both sides.

SASRec gets candidate discovery for **free** and scores only 512 candidates. Walker uses a native sparse terminal over 512 reachable products and never performs a full-catalog matmul in the timed path.

The notebook also evaluates validation + test NDCG in FP32 and BF16 from the actual checkpoints before timing. Fast forever-persistent KV/SWG shortcuts are excluded from the headline because both trained models use learned absolute positions and canonical 200-window recomputation semantics after saturation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, sys, shutil, subprocess, torch
REPO='/content/Sparsewalker'; BRANCH='agent/serving-speed-benchmark'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
sys.path.insert(0,f'{REPO}/src'); sys.path.insert(0,f'{REPO}/benchmarks'); sys.path.insert(0,f'{REPO}/experiments')
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


In [ ]:
import runpy
SCRIPT=f'{REPO}/benchmarks/run_final_quality_equivalent_serving.py'
sys.argv=[SCRIPT,'--catalog','1000000']
print('FINAL AUDIT START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('FINAL AUDIT END',flush=True)


Paste back `QUALITY`, `COMPILED_CORRECTNESS`, `CUDA_GRAPH`, and especially `HEADLINE`.


In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_speed/final_quality_equivalent_serving.json')
if p.exists():
    r=json.loads(p.read_text())
    print('HEADLINE',json.dumps(r['headline'],indent=2))
    print('QUALITY',json.dumps(r['quality'],indent=2))
    print('COMPILED_CORRECTNESS',json.dumps(r['compiled_correctness'],indent=2))
    print('CUDA_GRAPH',json.dumps(r['cuda_graph'],indent=2))
